# Treinando um modelo de Processamento de Linguagem Natural (NLP) utilizando Aprendizgem por Reforço -> Parte 2

Antes de tudo é nescessário fazer alguns comentários sobre o repositório que estamos utilizando:

- Este repositório é um fork **modificado** do repositório **nlp_gym** que implmenta um ambiente para treinamento de modelos de NLP utilizando Aprendizagem por reforço, através de um ambiente baseado e integrado na biblioteca Gym.

- Para o uso neste projeto algumas partes da estrutura do nlp_gym tiveram que ser modificadas, para resolver problemas de bibliotecas e funções deprecadas, assim como para ficarem compatíveis com os formatos mais recentes do Gym e da biblioteca Stable Baselines.

## Redes Neurais em Aprendizagem por Reforço

Relembrando um pouco da ultima parte, tradicionalmente, algoritmos de **aprendizagem por reforço** (RL), como o **Q-Learning**, usam tabelas (`Q-table`) para armazenar os valores Q de cada par estado-ação. Isso funciona bem em ambientes pequenos e discretos, como jogos simples ou problemas de grade, onde o número de estados e ações é limitado.

No entanto, muitos problemas do mundo real — incluindo aqueles em **NLP (Processamento de Linguagem Natural)**, **visão computacional** e **robótica** — envolvem **espaços de estados contínuos e de alta dimensionalidade**. Exemplos:
- Estados representados por *embeddings* de palavras ou sentenças (centenas ou milhares de dimensões).
- Estados como imagens ou sequências sensoriais complexas.
- Ações contínuas ou com alta cardinalidade.

Em tais cenários, armazenar explicitamente uma Q-table torna-se:
- **Impraticável** (devido à explosão combinatória).
- **Ineficiente** (pois não generaliza para estados similares).

### O papel das Redes Neurais

As **redes neurais** entram como **aproximadores de funções** em RL:
- Em vez de armazenar $Q(s, a)$ em uma tabela, a rede neural aprende uma função $Q(s, a; \theta)$, onde $\theta$ são os parâmetros da rede.
- A rede recebe como entrada o estado (e possivelmente a ação) e retorna os valores Q aproximados.
- Isso permite generalizar entre estados não vistos e lidar com entradas contínuas.

Essa abordagem é conhecida como:
- **Deep Reinforcement Learning (Deep RL)**.
- Um exemplo famoso é o **Deep Q-Network (DQN)**, e o agente jogador de Atari.


## Aprendizagem por Reforço utilizando o algorítimo DQN em um ambiente qualquer (ainda não NLP)

O **DQN** é um algoritmo de aprendizagem por reforço profundo (**Deep Reinforcement Learning**) que combina:
- O método clássico de **Q-Learning**  
- Redes neurais profundas como aproximadores de função

Ele foi popularizado pela DeepMind em 2015, quando mostrou desempenho acima do humano em diversos jogos do Atari, aprendendo **diretamente a partir de pixels**.

---

### Como funciona?

No Q-Learning clássico:
- Mantemos uma tabela $Q[s, a]$ que armazena o valor esperado de cada par estado-ação.

No DQN:
- Substituímos a Q-table por uma **rede neural** $Q(s, a; \theta)$, que recebe o estado como entrada e retorna os valores Q para todas as ações possíveis.

O treinamento usa:
- A equação de Bellman como alvo:
 
  $$y = r + \gamma \max_{a'} Q(s', a'; \theta^{-})$$
  
  Onde $\theta^{-}$ são os pesos da rede alvo (**target network**).

---

### Técnicas chave

DQN introduziu técnicas que tornaram o aprendizado estável:
 **Replay buffer**: armazena experiências passadas \((s, a, r, s')\) e treina a rede amostrando minibatches aleatórios → quebra correlações e melhora a eficiência.  
 **Rede alvo (target network)**: mantém uma cópia fixa da rede Q, atualizada periodicamente → estabiliza os alvos \( y \) e evita oscilações.  
 **Epsilon-greedy**: mantém exploração durante o treinamento, garantindo que o agente não fique preso em soluções subótimas.

---


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from tqdm import tqdm

# Define the Q-Network: a neural network that approximates Q(s, a)
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()
        # Camada totalmente conectada: entrada -> 64 neurônios
        self.fc1 = nn.Linear(input_dim, 64)
        # Camada totalmente conectada: 64 -> número de ações possíveis
        self.fc2 = nn.Linear(64, output_dim)

    def forward(self, x):
        # Passa pela primeira camada com ativação ReLU
        x = torch.relu(self.fc1(x))
        # Saída da rede: Q-values para cada ação
        x = self.fc2(x)
        return x

class DeepQLearning:
    def __init__(self, env, gamma, epsilon, epsilon_min, epsilon_dec, episodes,
                 batch_size, memory_size, max_steps, update_target_every=10):
        self.env = env
        self.gamma = gamma  # fator de desconto (quanto valorizamos recompensas futuras)
        self.epsilon = epsilon  # taxa de exploração inicial (epsilon-greedy)
        self.epsilon_min = epsilon_min  # mínimo de epsilon (exploração mínima)
        self.epsilon_dec = epsilon_dec  # fator de decaimento do epsilon
        self.episodes = episodes  # número total de episódios de treino
        self.batch_size = batch_size  # tamanho do minibatch para replay
        self.memory = deque(maxlen=memory_size)  # replay buffer
        self.max_steps = max_steps  # máximo de passos por episódio
        self.update_target_every = update_target_every  # frequência de atualização da rede alvo
        
        # Inicializa modelos: rede principal e rede alvo
        self.input_dim = env.observation_space.shape[0]
        self.output_dim = env.action_space.n
        self.model = QNetwork(self.input_dim, self.output_dim).to(device)
        self.target_model = QNetwork(self.input_dim, self.output_dim).to(device)
        self.target_model.load_state_dict(self.model.state_dict())  # sincroniza pesos inicialmente
        self.target_model.eval()  # rede alvo não é treinada diretamente
        
        # Otimizador e função de perda
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.002)
        self.loss_fn = nn.MSELoss()
        
        # Pré-alocação de tensores (para eficiência)
        self.states_tensor = None
        self.next_states_tensor = None
        self.actions_tensor = None
        self.rewards_tensor = None
        self.terminals_tensor = None

    def select_action(self, state):
        # Escolhe ação usando política epsilon-greedy
        if np.random.rand() < self.epsilon:
            return random.randrange(self.output_dim)  # exploração
        state_tensor = torch.FloatTensor(state).to(device)
        self.model.eval()
        with torch.no_grad():
            q_values = self.model(state_tensor)  # valores Q estimados
        action = torch.argmax(q_values, dim=1).item()  # escolhe ação com maior Q
        return action

    def experience_replay(self):
        # Só recomeça se já tiver amostras suficientes
        if len(self.memory) < self.batch_size:
            return None
            
        # Amostra minibatch aleatório do replay buffer
        batch = random.sample(self.memory, self.batch_size)
        
        # Extrai dados do batch
        states = np.vstack([exp[0] for exp in batch])
        actions = np.array([exp[1] for exp in batch])
        rewards = np.array([exp[2] for exp in batch])
        next_states = np.vstack([exp[3] for exp in batch])
        terminals = np.array([exp[4] for exp in batch]).astype(np.float32)
        
        # Constrói ou reutiliza tensores
        if self.states_tensor is None or self.states_tensor.shape[0] != states.shape[0]:
            self.states_tensor = torch.FloatTensor(states).to(device)
            self.next_states_tensor = torch.FloatTensor(next_states).to(device)
            self.actions_tensor = torch.LongTensor(actions).unsqueeze(1).to(device)
            self.rewards_tensor = torch.FloatTensor(rewards).unsqueeze(1).to(device)
            self.terminals_tensor = torch.FloatTensor(terminals).unsqueeze(1).to(device)
        else:
            self.states_tensor.copy_(torch.FloatTensor(states))
            self.next_states_tensor.copy_(torch.FloatTensor(next_states))
            self.actions_tensor.copy_(torch.LongTensor(actions).unsqueeze(1))
            self.rewards_tensor.copy_(torch.FloatTensor(rewards).unsqueeze(1))
            self.terminals_tensor.copy_(torch.FloatTensor(terminals).unsqueeze(1))

        # Calcula Q(s, a) atual usando rede principal
        self.model.train()
        q_values = self.model(self.states_tensor)
        
        # Implementação Double DQN: escolhe ação pela rede principal, valor pela rede alvo
        with torch.no_grad():
            next_actions = torch.argmax(self.model(self.next_states_tensor), dim=1, keepdim=True)
            next_q_values_target = self.target_model(self.next_states_tensor)
            next_q_values = torch.gather(next_q_values_target, 1, next_actions)
            
        # Calcula alvo: r + γ * max Q(s', a')
        targets = self.rewards_tensor + self.gamma * next_q_values * (1 - self.terminals_tensor)
        
        # Seleciona Q-values para ações realmente tomadas
        q_selected = torch.gather(q_values, 1, self.actions_tensor)
        
        # Calcula perda e faz backpropagation
        loss = self.loss_fn(q_selected, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()

    def train(self, early_stop_reward=-100, patience=50):
        rewards_all = []  # armazena recompensas totais por episódio
        losses = []  # armazena perdas médias por episódio
        best_avg_reward = -float('inf')
        episodes_without_improvement = 0
        
        for i in tqdm(range(self.episodes), desc="DQN"):
            state, _ = self.env.reset()
            state = np.reshape(state, (1, self.input_dim))
            score = 0
            steps = 0
            done = False
            episode_losses = []
            
            while not done:
                steps += 1
                action = self.select_action(state)
                next_state, reward, terminal, truncated, _ = self.env.step(action)

                # Pode ajustar recompensas para acelerar aprendizado
                shaped_reward = reward
                shaped_reward += 10 * abs(next_state[1])  # exemplo: penaliza velocidade
                
                if terminal or truncated or (steps > self.max_steps):
                    done = True
                    
                # Guarda experiência no replay buffer
                next_state = np.reshape(next_state, (1, self.input_dim))
                self.memory.append((state, action, shaped_reward, next_state, done))
                
                state = next_state
                score += reward
                
                # Treina rede com replay
                if len(self.memory) >= self.batch_size:
                    loss = self.experience_replay()
                    if loss is not None:
                        episode_losses.append(loss)
                
                if done:
                    # Decai epsilon (menos exploração ao longo do tempo)
                    if self.epsilon > self.epsilon_min:
                        self.epsilon *= self.epsilon_dec
                    break
            
            # Guarda métricas do episódio
            rewards_all.append(score)
            if episode_losses:
                losses.append(np.mean(episode_losses))
            
            # Atualiza rede alvo periodicamente
            if i % self.update_target_every == 0:
                self.target_model.load_state_dict(self.model.state_dict())
            
            # Checa condição de parada antecipada (early stopping)
            if i >= 100:
                current_avg_reward = np.mean(rewards_all[-100:])
                if current_avg_reward > best_avg_reward:
                    best_avg_reward = current_avg_reward
                    episodes_without_improvement = 0
                else:
                    episodes_without_improvement += 1
                
                if (current_avg_reward >= early_stop_reward or 
                    episodes_without_improvement >= patience):
                    print(f"\nDQN early stopping at episode {i+1}")
                    print(f"Average reward over last 100 episodes: {current_avg_reward:.2f}")
                    break
        
        return rewards_all, losses


### Próxima parte -> aplicando a NLP